<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Chapter 20 · Private Credit Analytics

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/ch20_private_credit_analytics.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

This notebook works a synthetic private-credit loan book end to end. You will

- compute lender coupon income on floating-rate loans as principal times base rate plus contractual spread,
- measure borrower capacity with leverage (debt over EBITDA) and interest coverage (EBITDA over interest),
- compute covenant headroom against maximum leverage and minimum coverage limits,
- stress the book with base-rate and EBITDA shocks and count covenant breaches per scenario, and
- contrast the two sides of the floating-rate trade: lender income that rises with base rates and borrower coverage that falls.

The loans are synthetic; the workflow and the verification discipline are the transferable part.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Run the chapter example

The cell below runs the Chapter 20 script. It builds the loan book with borrower covenant metrics, reports each borrower's coupon income, leverage, coverage, and covenant headroom, then evaluates the four stress scenarios borrower by borrower and aggregates breach counts. It closes with the borrower names attached to the frozen covenant text snippets.

In [ ]:
SCRIPT = ROOT / "code" / "ch20_private_credit_analytics.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

- Verify lender income as principal times (base rate plus spread): 800,000 for Atlas Software at 4.5% plus 5.5% on 8 million.
- Check that the base case already contains a breach: Cobalt Parts shows leverage headroom of -1.000 against its 5.0x maximum.
- Track the floating-rate tension: coupon income rises under base-rate shocks while the weakest coverage falls from 2.087 in the base case to 1.286 under severe stress.
- Note the pass-through assumption: only the sampled loan principal passes the base-rate shock into total borrower interest; all other interest is held fixed.
- Read the breach counts: leverage breaches rise from 1 to 3 and coverage breaches from 0 to 3 under severe stress.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch20_*.csv"))
summary = pd.DataFrame([
    {"file": path.name,
     "rows": len(frame := pd.read_csv(path)),
     "columns": len(frame.columns),
     "missing_cells": int(frame.isna().sum().sum()),
     "fields": str(list(frame.columns))}
    for path in chapter_files
])
summary


## Chapter-specific checks

Check both borrower leverage and interest-coverage covenants. Coupon income belongs to the lender; borrower coverage uses total borrower interest.

In [ ]:
loans = pd.read_csv(DATA / "ch20_private_credit_loans.csv")
metrics = pd.read_csv(DATA / "ch20_borrower_metrics.csv")
book = loans.merge(metrics, on="borrower")
book["coupon_income"] = book.principal * (book.base_rate + book.spread)
book["leverage"] = book.debt / book.ebitda
book["coverage"] = book.ebitda / book.interest
book["leverage_headroom"] = book.max_leverage - book.leverage
book["coverage_headroom"] = book.coverage - book.min_coverage
book["review_flag"] = (book.leverage_headroom.lt(0)
                       | book.coverage_headroom.lt(0))
book[["borrower", "coupon_income", "leverage",
      "coverage", "leverage_headroom", "coverage_headroom",
      "review_flag"]].round(3)

## Mini-lab: All-In Loan Yield Versus Deposit Funding Cost

A floating-rate loan earns the base rate plus the contractual spread, but the lender's economics also depend on the funding side. A book funded by deposits at 3.25% per year keeps the all-in yield minus the funding rate as its net margin. For the three synthetic borrowers, all-in yields run from 10.0% to 11.5%, so net margins run from 675 to 825 bp before default loss, fees, and administration costs. The wider margins belong to the weaker borrowers.

In [ ]:
import pandas as pd

funding_rate = 0.0325  # annual bank-deposit funding cost
loans = pd.DataFrame({
    "borrower": ["Atlas Software", "Beacon Health", "Cobalt Parts"],
    "principal": [8_000_000, 6_500_000, 5_000_000],
    "base_rate": [0.045, 0.045, 0.045],
    "spread": [0.055, 0.060, 0.070],
})
loans["all_in_yield"] = loans.base_rate + loans.spread
loans["net_spread_bp"] = (  # margin over deposit cost in bp
    loans.all_in_yield - funding_rate
) * 10_000
loans["coupon_income"] = loans.principal * loans.all_in_yield
loans["net_income"] = loans.principal * loans.all_in_yield
loans["net_income"] -= loans.principal * funding_rate
loans.round(2)

### What to observe

Atlas Software yields 10.0% all-in, a 675 bp margin over the 3.25% deposit cost, or 540,000 of net income on 8 million of principal. Beacon Health yields 10.5% (725 bp, 471,250 net) and Cobalt Parts 11.5% (825 bp, 412,500 net), for total net income of 1,423,750. The widest margin belongs to Cobalt Parts, the one borrower already breaching its leverage covenant in the base case.

## Exercises

1. **Coupon income by hand.** Compute annual coupon income for Beacon Health at a 4.5% base rate plus a 6.0% spread on 6.5 million of principal. *Hint:* 6,500,000 times 0.105.
2. **Covenant headroom.** Compute Cobalt Parts' leverage headroom from debt of 7.2 million, EBITDA of 1.2 million, and a 5.0x maximum. *Hint:* the base case breaches.
3. **Funding sensitivity.** Raise the deposit cost from 3.25% to 4.00% in the mini-lab and recompute total net income.
4. **Stress design.** Add a scenario with a 300 bp base-rate shock and a 20% EBITDA decline, and count the resulting breaches. *Hint:* mirror the pass-through assumption used by `code/ch20_private_credit_analytics.py`.
5. **Extraction check.** From `data/ch20_covenant_snippets.csv`, extract each borrower's leverage and coverage limits and verify them against the merged loan and borrower-metric tables.
6. **Verification rule.** State in two sentences why a model-extracted covenant level must be reconciled with the source clause before it enters a monitoring dashboard.

For more detail, see Chapter 20 of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">